In [8]:
import os
import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, OpenAIChatCompletionsModel, trace, function_tool, input_guardrail, GuardrailFunctionOutput

In [2]:
load_dotenv(override=True)

True

In [3]:
# Create Groq client (OpenAI-compatible)
client = AsyncOpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

In [5]:
# Attach model
model = OpenAIChatCompletionsModel(
    model="openai/gpt-oss-20b",   # Groq model name
    openai_client=client,
)

In [7]:
# Add guardrails
class CheckNameOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent(
    name="GuardRail Agent",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=CheckNameOutput,
    model=model
)

In [9]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [ ]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model=model,
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Past"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)